# 02 — Condensation: quench a vapor, watch droplets form

Take a hot 2D Lennard-Jones vapor and suddenly cool it below its
condensation point. The uniform gas becomes unstable: density fluctuations
grow, droplets nucleate everywhere, then bigger droplets eat smaller ones
(coarsening / Ostwald ripening). This is a genuine first-order phase
transition happening in your browser tab.

In [ ]:
%pip install lammps-js matplotlib

## A hot, uniform vapor

Density $\rho = 0.3$, temperature $T = 1.0$ — comfortably above the 2D LJ
critical temperature ($T_c \approx 0.5$), so the gas stays uniform. A weak
Langevin thermostat plays the role of the environment that carries heat
away. The `pe/atom` compute lets us color each atom by how tightly it is
bound — gas atoms ≈ 0, atoms inside a droplet strongly negative:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lammps import lammps, LMP_STYLE_ATOM, LMP_TYPE_VECTOR

lmp = await lammps(output=None)
lmp.commands_string("""
units         lj
dimension     2
lattice       sq 0.3
region        box block 0 40 0 40 -0.1 0.1
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 1.0 8712 dist gaussian
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
compute       peatom all pe/atom
fix           1 all nve
fix           lang all langevin 1.0 1.0 1.0 3216
fix           2d all enforce2d
thermo        1000
run           3000
""")

def snapshot():
    xy = lmp.extract_atom("x")[:, :2].copy()
    pe = lmp.extract_compute("peatom", LMP_STYLE_ATOM, LMP_TYPE_VECTOR).copy()
    return xy, pe

frames = [("hot vapor, T = 1.0", *snapshot())]
print(lmp.get_natoms(), "atoms equilibrated")

## Quench

Drop the thermostat target to $T = 0.4$ and keep integrating, saving
snapshots as the system condenses:

In [ ]:
lmp.command("fix lang all langevin 0.4 0.4 1.0 3216")
pe_trace, steps_trace = [], []
for label, steps in [("just after the quench", 2000),
                     ("droplets nucleate", 8000),
                     ("droplets coarsen", 20000)]:
    for _ in range(steps // 1000):
        lmp.command("run 1000")
        steps_trace.append(lmp.extract_global("ntimestep"))
        pe_trace.append(lmp.get_thermo("pe"))
    frames.append((label, *snapshot()))
print("final potential energy per atom:", round(pe_trace[-1], 3))

In [ ]:
L = 40 / np.sqrt(0.3)   # box edge in LJ units (lattice sq 0.3, 40 cells)
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for ax, (label, xy, pe) in zip(axes.flat, frames):
    sc = ax.scatter(xy[:, 0], xy[:, 1], c=pe, s=4, cmap="viridis_r",
                    vmin=-3.5, vmax=0.5)
    ax.set_title(label, fontsize=10)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, shrink=0.8, label="potential energy / atom")
plt.show()

Dark atoms are bound inside droplets; pale ones are still vapor. The
system's potential energy tracks the transition — flat while the vapor is
stable, then falling as bound pairs form:

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.plot(steps_trace, pe_trace)
plt.xlabel("timestep since quench"); plt.ylabel("potential energy / atom")
plt.title("Condensation releases latent heat")
plt.tight_layout(); plt.show()

lmp.close()

## Go bigger (optional)

More atoms means more droplets and better coarsening statistics. On a
cross-origin-isolated host this cell uses the multithreaded KOKKOS build
(see [basics/04](../basics/04-multithreading-kokkos.ipynb)); on the public
site it falls back to the single-threaded engine with a smaller box — still
entirely CPU-bound WebAssembly, so be patient with big boxes:

In [ ]:
import js
from lammps import lammps as lammps2

isolated = bool(getattr(js, "crossOriginIsolated", False))
n = 100 if isolated else 60
args = ["-k", "on", "t", "4", "-sf", "kk"] if isolated else None
big = await lammps2(cmdargs=args, output=None)
big.commands_string(f"""
units         lj
dimension     2
lattice       sq 0.3
region        box block 0 {n} 0 {n} -0.1 0.1
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 1.0 8712 dist gaussian
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
fix           lang all langevin 0.4 0.4 1.0 3216
fix           2d all enforce2d
thermo        2000
run           12000
""")
xy = big.extract_atom("x")[:, :2]
plt.figure(figsize=(6, 6))
plt.scatter(xy[:, 0], xy[:, 1], s=2)
plt.gca().set_aspect("equal"); plt.xticks([]); plt.yticks([])
plt.title(f"{big.get_natoms()} atoms, "
          f"{'KOKKOS ×4 threads' if isolated else 'single-threaded'}")
plt.tight_layout(); plt.show()
big.close()

Next: [03 — Real vs ideal gas](03-real-vs-ideal-gas.ipynb) — putting a
number on how non-ideal this gas is.